# E-Commerce Lakehouse: Bronze Orders


## Load the source orders

Read the original Olist orders CSV from our Unity Catalog Volume.

The first row contains column names, so we enable the header option.
We initially keep columns as strings to preserve the source values;
data types will be validated and converted in the Silver layer.

In [0]:
raw_orders_df = (
spark.read
.option("header", True)
.csv("/Volumes/workspace/ecommerce/raw_files/olist_orders_dataset.csv")
)

## Preview the orders data

Display five rows to inspect the column names and sample values before further processing.

In [0]:
display(raw_orders_df.limit(5))

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


## Inspect the source structure and row count

Count the source records and inspect the column data types.
These results establish a baseline for validating the Bronze table.

In [0]:
print("Source order count:", raw_orders_df.count())
raw_orders_df.printSchema()

Source order count: 99441
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: string (nullable = true)
 |-- order_approved_at: string (nullable = true)
 |-- order_delivered_carrier_date: string (nullable = true)
 |-- order_delivered_customer_date: string (nullable = true)
 |-- order_estimated_delivery_date: string (nullable = true)



## Add ingestion metadata

Preserve the original orders data and add an ingestion_timestamp
column to record when the data is processed by our pipeline.

In [0]:
from pyspark.sql.functions import current_timestamp

### Preview the Bronze orders

Inspect five rows to confirm that the original order fields remain
and the ingestion_timestamp column has been added.

In [0]:
bronze_orders_df = raw_orders_df.withColumn("ingestion_timestamp", current_timestamp())

### Preview the Bronze orders

Display five rows to inspect the original order data and the added
ingestion timestamp.

In [0]:
display(bronze_orders_df.limit(5))

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,ingestion_timestamp
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,2026-10-17T20:09:19.744Z
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,2026-10-17T20:09:19.744Z
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,2026-10-17T20:09:19.744Z
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,2026-10-17T20:09:19.744Z
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,2026-10-17T20:09:19.744Z


## Refresh the Bronze orders table

Rebuild workspace.ecommerce.bronze_orders from the complete source CSV.
Overwrite replaces existing contents instead of appending duplicate orders.
Verify that the saved row count matches the source.

In [0]:
table_name = "workspace.ecommerce.bronze_orders"

source_count = raw_orders_df.count()

if source_count != 99441:
    raise ValueError(
        f"Expected 99,441 source orders; found {source_count}. "
        "Stopped before replacing the table."
    )

(
    bronze_orders_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(table_name)
)

saved_count = spark.table(table_name).count()

if saved_count != source_count:
    raise ValueError("Saved Bronze count does not match the source.")

print("Bronze orders refreshed:", saved_count)

Bronze orders refreshed: 99441


## Validate the saved Bronze table

Read the saved Delta table and compare its row count with the
99,441 records in the source CSV.

In [0]:
saved_bronze_orders_df = spark.table("workspace.ecommerce.bronze_orders")

In [0]:
print(saved_bronze_orders_df.count())

99441
